# Small language models: afternoon exercises

Three short exercises, 10 to 15 minutes each, on this morning's bigram model. If you haven't yet, take the lecture's self-check quiz (`quiz.ipynb`) first. When you've done the exercises, it's on to the coursework.

<div class="alert alert-success">

**Why these exercises.** The coursework, released after this lecture, is a small package built test-first around today's ideas: counting, normalising, sampling with a seed, and log-likelihood and perplexity on held-out data. In it you'll need to:

- test code that uses random numbers (Exercise 1);
- choose settings, such as the amount of smoothing or the model's order, on held-out data (Exercise 2);
- check a suggestion (from an AI tool, say) with evidence before you accept it (Exercise 3).

Each exercise practises one of these on the lecture's own model, so that you've done it once before you need it.

</div>

<div class="alert alert-info">

### Aims:
 - Write `pytest` tests for numerical code: properties that must hold, reproducibility with a fixed seed, and a statistical check, each with a sensible tolerance
 - Choose settings (the amount of smoothing, and the model's order) on held-out data, and report a test score honestly
 - Check an AI suggestion with evidence, and record what you did in a SCOPE note

## Setup

Run this cell first. It rebuilds the pieces of the lecture these exercises use.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

import slmtools

text = slmtools.load_text("shakespeare")

chars = sorted(set(text))
V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}


def encode(s):
    return np.array([stoi[c] for c in s], dtype=np.int64)


def decode(ids):
    return "".join(itos[int(i)] for i in ids)


data = encode(text)
train_ids, val_ids, test_ids = data[:300_000], data[300_000:350_000], data[350_000:400_000]


# The order-m n-gram model from Part 4 of the lecture (for Exercise 2)
from collections import defaultdict


def fit_ngram(ids, order, V=V):
    '''Count how often each character follows each context of length `order`.'''
    table = defaultdict(lambda: np.zeros(V))
    for i in range(order, len(ids)):
        table[tuple(ids[i - order:i])][ids[i]] += 1
    return table


def ngram_nll(table, ids, order, k=1.0, V=V):
    '''Mean NLL of `ids`, plus the fraction of contexts that were seen in training.'''
    total, seen = 0.0, 0
    for i in range(order, len(ids)):
        counts = table.get(tuple(ids[i - order:i]))
        if counts is None:
            row = np.full(V, k)          # never seen this context at all
        else:
            row = counts + k
            seen += 1
        total -= np.log(row[ids[i]] / row.sum())
    n = len(ids) - order
    return total / n, seen / n


print(f"V = {V},  train {len(train_ids):,},  val {len(val_ids):,},  test {len(test_ids):,}")

---
## The bigram model as a module

To test code with `pytest`, it has to be importable. The cell below writes this morning's bigram model to `bigram.py`, in this folder: `bigram_counts` and `nll` as in the lecture, `bigram_probs` for the add-$k$ smoothed probabilities, and a `generate` that works with token ids and takes the random number generator as an argument.

In [ ]:
%%writefile bigram.py
"""This morning's bigram model, as functions that can be imported (and tested)."""

import numpy as np


def bigram_counts(ids, V):
    """Count adjacent pairs: N[i, j] is the number of times token j followed token i."""
    N = np.zeros((V, V), dtype=np.int64)
    np.add.at(N, (ids[:-1], ids[1:]), 1)
    return N


def bigram_probs(N, k=1.0):
    """Add-k smoothed probabilities: row i is the distribution of the token that follows token i."""
    return (N + k) / (N + k).sum(axis=1, keepdims=True)


def generate(P, start, n, rng):
    """Sample n more token ids from the bigram model P, starting from the id `start`."""
    out = [start]
    for _ in range(n):
        out.append(int(rng.choice(len(P), p=P[out[-1]])))
    return np.array(out)


def nll(P, ids):
    """Mean negative log-likelihood per token of `ids` under the bigram model P."""
    return -np.log(P[ids[:-1], ids[1:]]).mean()

In [ ]:
from bigram import bigram_counts, bigram_probs, generate, nll

---
## Exercise 1: testing the bigram model (about 15 minutes)

*Why this helps with the coursework:* the coursework is built test-first, and some of its code uses random numbers. Here you test code like that: properties that must always hold, reproducibility with a fixed seed, and a statistical check with a tolerance.

Write `pytest` tests for `bigram.py` in a file `test_bigram.py` (there's a starting point below). Small, hand-made inputs, where you know the right answer, make the best tests: they're quick, and when one fails it's easy to see why.

1. **A property that must hold.** For any counts and any $k > 0$, every row of `bigram_probs(N, k)` is a probability distribution: non-negative, and summing to 1. Why does the sum need a tolerance (`np.allclose`, or `pytest.approx`) rather than `==`?
2. **Reproducibility.** Two generators made with the same seed, `np.random.default_rng(seed)`, must give the same sample. (Would a test that two *different* seeds give different samples be a good one?)
3. **A statistical check.** Use a model where you know the answer, e.g. two tokens with `P = np.array([[0.25, 0.75], [0.5, 0.5]])`. Generate a long sample with a fixed seed, and check that token 1 follows token 0 about 75% of the time. How long a sample, and how wide a tolerance? (A proportion $p$ estimated from $n$ draws has a standard error of $\sqrt{p(1 - p)/n}$: roughly, how far the fraction you measure typically wanders from the true value by chance alone. It shrinks as the sample grows.) And why fix the seed?

Run the tests with `pytest test_bigram.py -v` in a terminal in this folder, or with the cell below. Then check that they can fail: change `axis=1` to `axis=0` in `bigram_probs`, run the `bigram.py` cell and the tests again, and then put it back.

In [ ]:
%%writefile test_bigram.py
"""Tests for bigram.py. Run with:  pytest test_bigram.py -v"""

import numpy as np
import pytest

from bigram import bigram_counts, bigram_probs, generate, nll


def test_rows_are_distributions():
    pytest.fail("not written yet")


def test_generate_is_reproducible():
    pytest.fail("not written yet")


def test_generate_follows_the_model():
    pytest.fail("not written yet")

In [ ]:
import sys

# The same as running `pytest test_bigram.py -v` in a terminal in this folder
!{sys.executable} -m pytest test_bigram.py -v

---
## Exercise 2: choosing settings on held-out text (about 15 minutes)

*Why this helps with the coursework:* in the coursework you'll choose settings, such as the amount of smoothing or the model's order, on held-out data, and report a score you can trust. Here you do both for the lecture's model.

In the lecture we used $k = 1$ throughout (a convenient choice, not a measured one), and with that $k$ the order sweep picked order 2. Choose them properly instead: fit on one part of the text, and judge on another.

1. **The smoothing.** Count the bigrams of `train_ids` once. For each $k$ in `(0.001, 0.01, 0.1, 0.3, 1, 3, 10)`, work out the perplexity of `bigram_probs(N, k)` on `train_ids` and on `val_ids`, and plot both against $k$ (with a log $x$-axis). Which $k$ would the training text choose, and which does the validation text choose? Why are they different?
2. **The order as well.** The lecture's `fit_ngram` and `ngram_nll` (in the setup cell) take an order and a $k$. Find the validation perplexity of orders 1, 2 and 3, each with $k$ in `(0.01, 0.1, 1)`. Which combination is best? Is it the order the lecture's sweep picked?
3. Evaluate the combination you chose on `test_ids`, once. Why only once, and only at the end?
4. Add a test to `test_bigram.py` that fits on one part of the corpus and evaluates on another: with $k > 0$ the held-out perplexity should be finite, and less than $V$ (better than a uniform guess). What does $k = 0$ give, and why?

If you have time: repeat step 1 with only the first 10,000 characters as training text. Does the best $k$ change?

In [ ]:
# Exercise 2

---
## Exercise 3: a SCOPE note (about 10 minutes)

*Why this helps with the coursework:* if you use AI tools for the coursework, SCOPE is a good way to check a suggestion before you accept it.

Below are two changes an AI assistant might suggest to this morning's code, written the way an assistant would write them: confident, well-commented and plausible. Some suggestions like these are right and some are wrong, and rejecting a good one is as much a failure as accepting a bad one.

Pick **one**, and write a SCOPE note on it, a few lines for each step:

| | |
|---|---|
| **S**tate | What is the code supposed to do? What are the constraints (shapes, dtypes, ranges, edge cases)? Write this down *before* you look hard at the suggestion. |
| **C**haracterise | What would the assistant have needed to know to get this right, and did it have that? (It hasn't seen your tests, your data or the shapes at run time unless you showed it.) |
| **O**btain | What exactly does the suggestion *claim*? "It's faster", "it gives the same result" and "it's more idiomatic" are different claims, and they need different evidence. |
| **P**robe | **Write a test or a measurement that would tell true from false, and run it.** Reading the code carefully isn't enough on its own. |
| **E**valuate | Accept, reject or revise, and say why, pointing to your evidence. |

Your reasoning matters more than your verdict.

### Suggestion A

> *"`np.add.at` is notoriously slow - it's an unbuffered ufunc call and it doesn't vectorise. You can get the same result with plain fancy-index assignment, which is a straightforward speed-up of an order of magnitude."*

In [ ]:
def bigram_counts_suggestion_a(ids, V):
    '''Count adjacent pairs. N[i, j] = number of times j followed i.'''
    N = np.zeros((V, V), dtype=np.int64)
    N[ids[:-1], ids[1:]] += 1
    return N

### Suggestion B

> *"Your `encode` builds a Python list one character at a time, which is slow for a corpus of this size. Since the vocabulary is ASCII, you can build a 256-entry lookup table once and then index into it with the raw bytes - it's the same result, vectorised."*

In [ ]:
_LOOKUP = np.zeros(256, dtype=np.int64)
for _c, _i in stoi.items():
    _LOOKUP[ord(_c)] = _i

def encode_suggestion_b(s):
    '''str -> np.ndarray of token ids.'''
    return _LOOKUP[np.frombuffer(s.encode("ascii"), dtype=np.uint8)]

### Your SCOPE note

Use the cell below for your evidence (the code you ran), and the one after it for the note.

In [ ]:
# Exercise 3: evidence

**Suggestion:** A or B

**State:**

**Characterise:**

**Obtain:**

**Probe:**

**Evaluate:**